# Spark execution
Today we will dive into the execution internals of spark. We will be reading some (fake) transactions and perform some transformations on them. While doing these transformations, we aim to get more insights into how spark works. We will check wide and narrow transformations, shuffles and spark plans to understand what is happening behind the code.

## First we will import the pyspark library and create a spark session

In [ ]:
from pyspark.sql import SparkSession, functions as sf

In [ ]:
spark = SparkSession.builder.getOrCreate()

## Now we will read the data and we will show the first records

In [ ]:
df = spark.read.csv("transactions.csv")

In [ ]:
df.show()

Do you recognize what just happened? Remember lazy evaluation and actions?

We called show, an action, and spark started to execute.

## Make a column where we want to check if this is a domestic transaction

In [ ]:
df = df.withColumn(
    "is_domestic",
    ...
)

## We want to only keep eur and usd transactions, create a filter for that

In [ ]:
df_eur_usd = df.where(
    ...
)

## Let's check how spark executes this

In [ ]:
df_eur_usd.explain()

## What do you see?

## What happens when we filter on only domestic transactions?

In [ ]:
df_domestic = df.where(
    ...
)

In [ ]:
df_domestic.explain()

## We want to convert the country codes to full country names

Run the code below to prevent spark from converting your join to broadcast hash joins

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

First, read in the conversion table. Then perform a left join.

In [ ]:
country_df = ...
df_smj = df.join(
    country_df,
    ...
)

In [ ]:
df.explain()

What do you notice?

In [ ]:
df_bhj = df.join(
    sf.broadcast(country_df),
    ...
)

Does this plan look different than before? Which one do you think will be faster?

## Now we would like to know the total amounts transferred for each currency

In [ ]:
total_currency = df.groupBy(
    ...
)

In [ ]:
total_currency.show()

In [ ]:
total_currency.explain()

What do you notice?